In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip install -q pycocotools scipy

In [9]:
# Upload garment-panel-seg.zip as a Kaggle Dataset first (Add Input > New Dataset),
# then this finds and unzips it. Falls back to git clone if you pushed to GitHub.
import os, glob, zipfile, shutil

WORK = "/kaggle/working/garment-panel-seg"
if os.path.exists(WORK):
    shutil.rmtree(WORK)

zips = glob.glob("/kaggle/input/**/garment-panel-seg*.zip", recursive=True)
if zips:
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall("/kaggle/working/")
    print("unzipped repo from", zips[0])
else:
    # fallback: replace with your GitHub URL once pushed
    !cd /kaggle/working && git clone https://github.com/Universalmike/garment-panel-seg.git
os.chdir(WORK)
print("cwd:", os.getcwd())
print(os.listdir("."))

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
chdir: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
Cloning into 'garment-panel-seg'...
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 22 (delta 0), reused 22 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (22/22), 22.30 KiB | 22.30 MiB/s, done.
cwd: /kaggle/working/garment-panel-seg
['tests', 'src', 'predict.py', 'example_apply_fabric.py', 'DESIGN_NOTE.md', 'example_before.png', '.git', 'README.md', 'example_after.png', 'requirements.txt', 'weights', '.gitignore']


In [10]:
import glob, os, subprocess

# 1) look for full annotations already added as a Kaggle input
found = glob.glob("/kaggle/input/**/instances_attributes_*2020.json", recursive=True)

FP = "/kaggle/working/fp"
os.makedirs(FP, exist_ok=True)

if found:
    ANN = found[0]
    # find the image dir that goes with it: the folder holding the most jpgs
    cand = {}
    for p in glob.glob("/kaggle/input/**/*.jpg", recursive=True):
        d = os.path.dirname(p); cand[d] = cand.get(d, 0) + 1
    IMGS = max(cand, key=cand.get)
    print("using existing annotations:", ANN)
else:
    # official CVDF download (validation set: small + fast, has the apparel parts)
    ANN = f"{FP}/instances_attributes_val2020.json"
    print("downloading official Fashionpedia val set from CVDF...")
    !wget -q -O {ANN} https://s3.amazonaws.com/ifashionist-dataset/annotations/instances_attributes_val2020.json
    !wget -q -O {FP}/val_test2020.zip https://s3.amazonaws.com/ifashionist-dataset/images/val_test2020.zip
    !cd {FP} && unzip -q -o val_test2020.zip
    cand = {}
    for p in glob.glob(f"{FP}/**/*.jpg", recursive=True):
        d = os.path.dirname(p); cand[d] = cand.get(d, 0) + 1
    IMGS = max(cand, key=cand.get)

# ---- to use the FULL train set instead (better model, ~20GB+ download), swap the
#      two wget lines above for these and set ANN to the train json:
# https://s3.amazonaws.com/ifashionist-dataset/annotations/instances_attributes_train2020.json
# https://s3.amazonaws.com/ifashionist-dataset/images/train2020.zip

print("ANN :", ANN)
print("IMGS:", IMGS, "| jpg count:", cand[IMGS])

downloading official Fashionpedia val set from CVDF...
ANN : /kaggle/working/fp/instances_attributes_val2020.json
IMGS: /kaggle/working/fp/test | jpg count: 3200


In [8]:
import glob
print(glob.glob("/kaggle/working/**/src/prepare_data.py", recursive=True))

[]


In [11]:
!python -m src.prepare_data \
    --ann  "{ANN}" \
    --imgs "{IMGS}" \
    --out  /kaggle/working/data/train \
    --max-images 4000 --require-sleeve --seed 42

loading annotations...
category ids -> body:[0, 1, 2, 3, 4, 5, 9, 10, 11, 12] sleeve:[31] collar:[28]
100%|██████████████████████████████████████| 1158/1158 [00:10<00:00, 108.19it/s]
wrote 742 mask/image pairs -> /kaggle/working/data/train/manifest.json


In [12]:
!python -m src.train \
    --manifest /kaggle/working/data/train/manifest.json \
    --epochs 30 --size 256 --batch 16 --seed 42 \
    --out /kaggle/working/garment-panel-seg/weights/model.pt

device: cuda
train 631  val 111
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth
100%|███████████████████████████████████████| 13.6M/13.6M [00:00<00:00, 228MB/s]
trainable params 491,564  |  total inference params 2,303,276
epoch 01/30  loss 1.1548  val_mIoU 0.4370
  saved /kaggle/working/garment-panel-seg/weights/model.pt (val_mIoU 0.4370)
epoch 02/30  loss 0.7548  val_mIoU 0.4813
  saved /kaggle/working/garment-panel-seg/weights/model.pt (val_mIoU 0.4813)
epoch 03/30  loss 0.6800  val_mIoU 0.4863
  saved /kaggle/working/garment-panel-seg/weights/model.pt (val_mIoU 0.4863)
epoch 04/30  loss 0.6130  val_mIoU 0.5075
  saved /kaggle/working/garment-panel-seg/weights/model.pt (val_mIoU 0.5075)
epoch 05/30  loss 0.5727  val_mIoU 0.5378
  saved /kaggle/working/garment-panel-seg/weights/model.pt (val_mIoU 0.5378)
epoch 06/30  loss 0.5236  val_mIoU 0.5361
epoch 07/30  loss 0.5049  val_mIoU 0.5482
  sav

In [13]:
import os
w = "/kaggle/working/garment-panel-seg/weights/model.pt"
print("weights saved:", os.path.exists(w), "| size MB:",
      round(os.path.getsize(w)/1e6, 1) if os.path.exists(w) else "-")

# quick visual check on one val image
import glob
img = glob.glob(f"{IMGS}/*.jpg")[0]
!cd /kaggle/working/garment-panel-seg && python predict.py --image "{img}" --out /kaggle/working/pred_mask.png

weights saved: True | size MB: 9.5
wrote /kaggle/working/pred_mask.png
panels present: ['front_body', 'left_sleeve', 'right_sleeve']
latency: 372.7 ms/image on CPU (x86_64)


# Evaluation — score the shipped checkpoint the way the brief scores it

Everything above is the original training run. This section does **not** retrain:
the trained checkpoint is committed to the repo, so we only need the dataset back
in order to score it.

**Before running this, push your local commits to GitHub.** These cells clone the
repo from `github.com/Universalmike/garment-panel-seg`, so an unpushed local
commit will silently give you the old code. The next cell checks for
`src/evaluate.py` and fails loudly if you forgot.

Runtime is a few minutes: the val/test image zip is the only slow part, and no
GPU is needed.

In [ ]:
!pip install -q pycocotools scipy pytest

import os, shutil

os.chdir("/kaggle/working")                     # never rmtree the cwd
WORK = "/kaggle/working/garment-panel-seg"
if os.path.exists(WORK):
    shutil.rmtree(WORK)

!git clone -q https://github.com/Universalmike/garment-panel-seg.git
os.chdir(WORK)

print("cwd:", os.getcwd())
!git log --oneline -1

# Guards. If either of these is False you are running stale code: push and re-run.
print("src/evaluate.py present :", os.path.exists("src/evaluate.py"))
print("src/metrics.py present  :", os.path.exists("src/metrics.py"))
print("weights/model.pt present:", os.path.exists("weights/model.pt"),
      "|", round(os.path.getsize("weights/model.pt") / 1e6, 1) if os.path.exists("weights/model.pt") else "-", "MB")

assert os.path.exists("src/evaluate.py"), "stale clone - push your commits to GitHub first"
assert os.path.exists("weights/model.pt"), "checkpoint missing from the clone" 

In [ ]:
# Fashionpedia val2020. If you have attached it as a Kaggle input dataset this
# reuses it; otherwise it pulls from the official CVDF mirror (~1 GB, a few min).
import glob, os

found = glob.glob("/kaggle/input/**/instances_attributes_*2020.json", recursive=True)
FP = "/kaggle/working/fp"
os.makedirs(FP, exist_ok=True)

if found:
    ANN = found[0]
    print("using attached annotations:", ANN)
else:
    ANN = f"{FP}/instances_attributes_val2020.json"
    if not os.path.exists(ANN):
        !wget -q -O {ANN} https://s3.amazonaws.com/ifashionist-dataset/annotations/instances_attributes_val2020.json
    if not glob.glob(f"{FP}/**/*.jpg", recursive=True):
        !wget -q -O {FP}/val_test2020.zip https://s3.amazonaws.com/ifashionist-dataset/images/val_test2020.zip
        !cd {FP} && unzip -q -o val_test2020.zip

# the image folder is whichever directory holds the most jpgs
cand = {}
roots = ["/kaggle/input"] if found else [FP]
for root in roots:
    for jpg in glob.glob(f"{root}/**/*.jpg", recursive=True):
        d = os.path.dirname(jpg)
        cand[d] = cand.get(d, 0) + 1
IMGS = max(cand, key=cand.get)

print("ANN :", ANN)
print("IMGS:", IMGS, "| jpg count:", cand[IMGS])

In [ ]:
# Rebuild the exact same manifest the model was trained on. prepare_data shuffles
# with --seed, so seed 42 on the same annotation file reproduces the same 742
# pairs in the same order - which is what makes the val split below the genuine
# held-out set rather than images the model has already seen.
!python -m src.prepare_data \
    --ann  "{ANN}" \
    --imgs "{IMGS}" \
    --out  /kaggle/working/data/train \
    --max-images 4000 --require-sleeve --seed 42

In [ ]:
# Quick health check on the clone before trusting any numbers it produces.
!pytest -q

## The actual measurement

`--val-split` and `--seed` must match what `train.py` used (0.15 and 42), so this
scores exactly the 111 images the model never saw.

`--compare` runs it twice, with and without horizontal-flip TTA. That decides
whether TTA ships enabled — it is currently **off** by default because it was
unmeasured, and a synthetic probe suggested it hurt.

In [ ]:
!python -m src.evaluate \
    --manifest /kaggle/working/data/train/manifest.json \
    --weights  weights/model.pt \
    --val-split 0.15 --seed 42 \
    --compare

## What to do with that output

1. **Paste the per-class table into the README**, under *Results (validation
   set)*, replacing the note that says to run this. Those are the honest
   per-panel numbers; the 0.572 in the training log above is measured with the
   old batch-aggregated, background-inclusive metric and is an upper bound.

2. **If TTA won**, flip `USE_TTA_BY_DEFAULT = True` in `predict.py` and say so in
   the README. If it lost, leave it off and quote the number — a measured
   decision to disable something reads better than silence either way.

3. Expect `collar` to be the weakest class. It is small, thin, and boundary
   error dominates its IoU.

## Optional — retrain

Only needed if you change the model or the data. The checkpoint in the repo is
already trained; this cell exists so the run is reproducible end to end.

Note that `train.py` now selects the best checkpoint on the **panels-only,
per-image** metric rather than the background-inclusive one, so it may pick a
different epoch than the original run did — that is the intended fix, not drift.

To train on the full `train2020` split instead (~45k images, the single biggest
quality lever), attach Fashionpedia as a Kaggle **input dataset** rather than
downloading it: inputs do not count against the 20 GB working quota, and the
cell above already picks up attached annotations automatically.

In [ ]:
!python -m src.train \
    --manifest /kaggle/working/data/train/manifest.json \
    --epochs 30 --size 256 --batch 16 --seed 42 \
    --out /kaggle/working/garment-panel-seg/weights/model.pt

## Download the retrained checkpoint

Kaggle sessions are ephemeral. If you retrained above, save the result out and
commit it to the repo, otherwise the run is lost when the session ends.

In [ ]:
import os
w = "/kaggle/working/garment-panel-seg/weights/model.pt"
print("exists:", os.path.exists(w), "| MB:", round(os.path.getsize(w) / 1e6, 1))

import torch
ck = torch.load(w, map_location="cpu")
for k in ("val_miou", "val_miou_with_background", "per_class_iou", "size", "seed"):
    if k in ck:
        print(f"{k:>26}: {ck[k]}")

# copy to /kaggle/working root so it appears in the session's Output tab
import shutil
shutil.copy(w, "/kaggle/working/model.pt")
print("\ncopied to /kaggle/working/model.pt - download it from the Output tab")

---

# Synthetic renders — closing the domain gap

Fashionpedia is photographs of worn garments. Production images are 3D renders of
a garment on an invisible form against a clean backdrop. Probing the
Fashionpedia-trained model on render-style inputs had it predicting background
across the whole garment at ~0.99 confidence, so this is not a small shift.

`src/synth.py` procedurally generates garment renders with pixel-exact labels:
directional lighting, two octaves of fold shading, per-panel ambient occlusion at
the seams, woven texture, varied colourways and proportions, a contact shadow,
and — deliberately — sleeveless and collarless garments, because the brief's
held-out set contains a garment with an absent panel.

It is procedural drawing plus a shading model. No diffusion, no hosted image
model: nothing the brief rules out.

**Run the Fashionpedia cells above first** — the comparison at the end needs the
real manifest to score against.

In [ ]:
# ~5 min on Kaggle for 2000 samples.
!python -m src.synth \
    --out  /kaggle/working/data/synth \
    --n    2000 \
    --size 384 \
    --seed 42 \
    --contact-sheet /kaggle/working/synth_sheet.png

### Look at it before training on it

This is not optional. Synthetic data that looks wrong teaches the model something
wrong, and the failure is silent — you would only see it as a disappointing IoU
several hours later. Top row of each pair is the render, bottom row is its mask
(blue body, orange sleeve, red collar).

Things that would mean stop and fix the generator: sleeves detached from the
shoulder, masks not lining up with the garment, every garment looking identical,
or the neck reading as a hole punched through to the backdrop rather than the
shadowed inside of the garment.

In [ ]:
from IPython.display import Image as IPyImage
IPyImage("/kaggle/working/synth_sheet.png", width=950)

### The mix

Synthetic goes into **training only**. The validation split stays pure
Fashionpedia, on purpose: synthetic images are far easier than photographs, so
letting them into validation would inflate the number and make it incomparable
to the run above.

`--synth-frac 0.5` of 2000 gives ~1000 synthetic against 631 real, so roughly
60/40 synthetic. That is a starting point, not a tuned value — it is the obvious
thing to sweep if you have time, and the honest thing to say if you do not.

In [ ]:
!python -m src.train \
    --manifest       /kaggle/working/data/train/manifest.json \
    --synth-manifest /kaggle/working/data/synth/manifest.json \
    --synth-frac 0.5 \
    --epochs 30 --size 256 --batch 16 --seed 42 \
    --out /kaggle/working/garment-panel-seg/weights/model_mixed.pt

### Does it hold up on real photos?

Both checkpoints scored on the **same** real held-out split. This answers the
question we can actually answer: did adding render-style data cost us anything on
photographs?

What it does **not** answer is whether the model got better on renders — that
needs render samples we do not have. Their held-out set is the real test. So a
mixed model that roughly matches the baseline on photos is a good outcome, not a
disappointing one: it means we bought render coverage without paying for it.

In [ ]:
print("=" * 30, "BASELINE (Fashionpedia only)", "=" * 30)
!python -m src.evaluate \
    --manifest /kaggle/working/data/train/manifest.json \
    --weights  weights/model.pt \
    --val-split 0.15 --seed 42

print()
print("=" * 30, "MIXED (Fashionpedia + synthetic)", "=" * 30)
!python -m src.evaluate \
    --manifest /kaggle/working/data/train/manifest.json \
    --weights  /kaggle/working/garment-panel-seg/weights/model_mixed.pt \
    --val-split 0.15 --seed 42

### Deciding

- **Mixed roughly matches or beats baseline on photos** → promote it. You gained
  render-domain coverage for free.
- **Mixed is clearly worse on photos** → lower `--synth-frac` and retrain, or keep
  the baseline. Say which you did and why in the README; a measured decision to
  reject synthetic data is a perfectly good result to report.

Whichever you pick, put both numbers in the README. Showing the comparison is
worth more than showing the winner.

In [ ]:
import shutil, os, torch

MIXED = "/kaggle/working/garment-panel-seg/weights/model_mixed.pt"
SHIP  = "/kaggle/working/garment-panel-seg/weights/model.pt"

# Uncomment to promote the mixed model to the shipped checkpoint, then commit it.
# shutil.copy(MIXED, SHIP)

for name, path in [("baseline", SHIP), ("mixed", MIXED)]:
    if os.path.exists(path):
        ck = torch.load(path, map_location="cpu")
        print(f"{name:>9}: val_mIoU {ck.get('val_miou'):.4f} | {round(os.path.getsize(path)/1e6,1)} MB")
        if "per_class_iou" in ck:
            print(" " * 11, ck["per_class_iou"])

# copy out so it survives the session ending
if os.path.exists(MIXED):
    shutil.copy(MIXED, "/kaggle/working/model_mixed.pt")
    print("\ndownload /kaggle/working/model_mixed.pt from the Output tab")